### *Install dependencies (uncomment if needed)*

In [1]:
# !pip install requests
# !pip install beautifulsoup4
# !pip install psycopg2

### Imports

In [2]:
import requests
from bs4 import BeautifulSoup
import time, random
import re
import psycopg2
from psycopg2.extras import execute_values
import json

## Fetch page

In [3]:
# Set up headers and target URL
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/116.0.5845.111 Safari/537.36"
}
url = 'https://en.wikipedia.org/wiki/List_of_highest-grossing_films'

# Fetch the Wikipedia page
page = requests.get(url, headers=headers)

## Parse HTML

In [ ]:
soup = BeautifulSoup(page.text, "html.parser")
# Find all tables with class "wikitable"
allTables = soup.find_all('table', class_='wikitable')

# Find the target table by caption text
target_table = None
for table in allTables:
    caption = table.find("caption")
    if caption:
        # Remove any superscript references
        for sup in caption.find_all("sup"):
            sup.decompose()
        if caption.get_text(strip=True).lower() == "highest-grossing films":
            target_table = table

# Extract table rows
rows = target_table.find_all("tr")
films = []

# Session object for repeated requests
session = requests.Session()

for row in rows[1:]:   # skip header row
    col = row.find_all("td")
    if len(col) < 5:   # skip incomplete rows
        continue
    gross = col[2].text.strip()   # box office
    year = col[3].text.strip()   # release year

    title_link = None
    title_cell = row.find("th", scope="row")
    if title_cell:
        title_link = title_cell.find("a")

    director = genre = country = None
    if title_link:
        title = title_link.text.strip()
        film_url = "https://en.wikipedia.org" + title_link.get("href")

        # Fetch individual film page
        film_page = session.get(film_url, headers=headers)
        if film_page.status_code != 200:
            while film_page.status_code != 200:
                film_page = session.get(film_url, headers=headers)
                time.sleep(5)

        director = country = None

        # Parse film info from infobox
        film_soup = BeautifulSoup(film_page.text, "html.parser")
        table_film_info = film_soup.find('table', class_="infobox vevent")

        for tr in table_film_info.find_all("tr"):
            th = tr.find("th")
            td = tr.find("td")
            if th and td:
                key = th.text.strip()
                plainlist = td.find("div", class_="plainlist")
                if plainlist:
                    items = [li.get_text(strip=True) for li in plainlist.find_all("li")]
                else:
                    items = [td.get_text().strip()]
                # Extract directors
                if "Directed by" in key:
                    director = items
                # Extract countries
                elif "Country" in key or "Countries" in key:
                    country = items
                    
        # Random delay to avoid overloading Wikipedia
        time.sleep(random.uniform(2, 5))
        
    films.append({
        "title": title,
        "year": year,
        "box_office": gross,
        "director": director,
        "country": country
    })


## Clean data

In [ ]:
for film in films:
    # Extract year as integer
    film["year"] = int(re.findall(r"\d{4}", str(film["year"]))[0])

    # Clean box office string to integer
    revenue = film["box_office"]
    revenue_clean = re.sub(r"[^\d]", "", revenue)
    film["box_office"] = int(revenue_clean)

    # Remove references and whitespace from directors
    cleaned_directors = []
    for d in film.get("director", []):
        d_clean = re.sub(r"\[.*?\]", "", d).strip()
        if d_clean:
            cleaned_directors.append(d_clean)
    film["director"] = cleaned_directors

    # Remove references and whitespace from countries
    cleaned_countries = []
    for c in film.get("country", []):
        c_clean = re.sub(r"\[.*?\]", "", c).strip()
        if c_clean:
            cleaned_countries.append(c_clean)
    film["country"] = cleaned_countries


## Save to Database

In [ ]:
conn = psycopg2.connect(
    database='highest_grossing_films',
    user='postgres',
    password='postgres',
    host='localhost',
    port='5432'
)

cursor = conn.cursor()

# Prepare list of tuples for insertion
values = [
    (
        f["title"],
        f["year"],
        f["director"],
        f["box_office"],
        f["country"]
    )
    for f in films
]

# Insert all films using UPSERT (ignore duplicates)
sql = "INSERT INTO films (title, release_year, director, box_office, country) VALUES %s ON CONFLICT ON CONSTRAINT unique_films DO NOTHING;"
execute_values(cursor, sql, values)

conn.commit()
cursor.close()
conn.close()

## Export to JSON

In [ ]:
conn = psycopg2.connect(
    database='highest_grossing_films',
    user='postgres',
    password='postgres',
    host='localhost',
    port='5432'
)
cursor = conn.cursor()

# Fetch all rows
cursor.execute("SELECT title, release_year, director, box_office, country  FROM films;")
rows = cursor.fetchall()

# Convert rows to list of dictionaries
films_json = []
for row in rows:
    film = {
        "title": row[0],
        "year": row[1],
        "director": row[2],
        "box_office": row[3],
        "country": row[4]
    }
    films_json.append(film)

# Save to JSON file
with open("films.json", "w", encoding="utf-8") as f:
    json.dump(films_json, f, ensure_ascii=False, indent=4)